In [2]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"

In [3]:
from generate_subqueries_agent import run_generate_subqueries_agent
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, SystemMessagePromptTemplate
from langchain_core.messages import HumanMessage
from multi_agent import llm
from pydantic import BaseModel

In [4]:
class GradeSubQuery(BaseModel):
    score: int

In [5]:
llm_grade = llm.with_structured_output(GradeSubQuery)
def run_test(query: str):
    sub_queries = run_generate_subqueries_agent({"messages": [HumanMessage(content=query)]})
    system_prompt = """
    Bạn là một giám kháo chấm điểm độ liên quan của các truy vấn con đối với truy vấn chính. 
    Điểm số từ 1 đến 10, trong đó 10 là rất liên quan và 1 là không liên quan.
    Yêu cầu: Đánh giá một cách gắt gao, công tâm và chính xác.
    """
    prompt = ChatPromptTemplate.from_messages([
        SystemMessagePromptTemplate.from_template(system_prompt),
        HumanMessagePromptTemplate.from_template(
            "Query chính: {query}\n"
            "Query phụ: {sub_query}\n"
        )
    ])
    llm_judge = prompt | llm_grade
    scores = []
    for sub_query in sub_queries[1:]:
        response = llm_judge.invoke({"query": query, "sub_query": sub_query})
        scores.append(response.score)
    print(f"Query: {query}")
    average_score = sum(scores) / len(scores) if scores else 0
    print(f"Average scores: {average_score}")
    return sub_queries, average_score


In [6]:
run_test("Giải thích Quốc Hội Việt Nam là gì?")

Query: Giải thích Quốc Hội Việt Nam là gì?
Average scores: 9.8


(['Giải thích Quốc Hội Việt Nam là gì?',
  'Quốc hội Việt Nam được quy định tại văn bản pháp luật nào?',
  'Quốc hội Việt Nam là cơ quan gì, có vị trí, vai trò và chức năng, nhiệm vụ chính nào trong hệ thống chính trị Việt Nam?',
  'Cơ cấu tổ chức của Quốc hội Việt Nam bao gồm những thành phần nào và nhiệm kỳ của Quốc hội là bao lâu?',
  'Quốc hội Việt Nam có những quyền hạn và nhiệm vụ cụ thể nào trong việc lập hiến, lập pháp, quyết định các vấn đề quan trọng của đất nước và giám sát tối cao?',
  'Đại biểu Quốc hội được bầu cử theo nguyên tắc và quy định nào?'],
 9.8)

In [ ]:
import pandas as pd
data = pd.read_csv("E:\Downloads\projectcongty\Dataset\sent_truncated_dvc_train.csv", encoding='utf-16')
data = data.sample(frac=1, random_state=42)

In [ ]:
print(questions_test[2])

In [ ]:
questions = data["noi_dung_hoi"].tolist()
scores_test = []
sub_queries_list = []
questions_test = questions[:20]
for query in questions_test:
    sub_queries, score = run_test(query)
    sub_queries_list.append(sub_queries)
    scores_test.append(score)
    print("-----")

average_test_score = sum(scores_test) / len(scores_test) if scores_test else 0
print(f"Average test scores: {average_test_score}")
#lưu sub_queries_list và questions_test vào file để dùng sau
import pickle
with open("sub_queries_list.pkl", "wb") as f:
    pickle.dump(sub_queries_list, f)
with open("questions_test.pkl", "wb") as f:
    pickle.dump(questions_test, f)

In [ ]:
import os
import sys
import transformers.utils.import_utils

# Patch: Đánh lừa transformers rằng TensorFlow không tồn tại
# Cách này tránh lỗi "DLL load failed" và lỗi "None in sys.modules"
os.environ["TRANSFORMERS_NO_TF"] = "1"
transformers.utils.import_utils._tf_available = False

#compute cosine similarity between two vectors
from sklearn.metrics.pairwise import cosine_similarity
def compute_cosine_similarity(vec1, vec2):
    return cosine_similarity([vec1], [vec2])[0][0]

PATH_TO_EMBEDDING = "intfloat/multilingual-e5-large"
from sentence_transformers import SentenceTransformer
# Thêm trust_remote_code=True vì model này yêu cầu code custom
embedder = SentenceTransformer(PATH_TO_EMBEDDING, trust_remote_code=True)
def get_embedding(text: str):
    embedding = embedder.encode(text)
    return embedding
# Example usage
vec1 = get_embedding("This is a sample sentence.")
print(vec1)

In [ ]:
print(get_similarity(questions_test[1], sub_queries_list[1][3]))

0.8883544


In [ ]:
def get_similarity(main_query: str, sub_query:str):
    similarity = compute_cosine_similarity(get_embedding(main_query), get_embedding(sub_query))
    return similarity
def get_average_similary(main_query: str, sub_queries: list[str]):
    similarities = [get_similarity(main_query, sub_query) for sub_query in sub_queries[1:]]
    average_similarity = sum(similarities) / len(similarities) if similarities else 0
    return average_similarity
similarities_test = []
for query, sub_queries in zip(questions_test,sub_queries_list):
    avg_sim = get_average_similary(query, sub_queries)
    similarities_test.append(avg_sim)
    print(f"Query: {query}")
    print(f"Average similarity: {avg_sim}")
    print("-----")
print(f"Average similarity on test set: {sum(similarities_test)/len(similarities_test)}")

Query: Để xác định trước mã số cần chuẩn bị hồ sơ như thế nào?
Average similarity: 0.9307035207748413
-----
Query: Xin cho hỏi thời hạn thanh toán trực tiếp chi phí khám, chữa bệnh bảo hiểm y tế tại cơ quan bảo hiểm xã hội là bao lâu?
Average similarity: 0.90125572681427
-----
Query: Thời điểm lập Hồ sơ cấp giấy phép của các công trình thuộc đối tượng lấy ý kiến đại diện cộng đồng dân cư, tổ chức, cá nhân liên quan trong khai thác, sử dụng tài nguyên nước sau khi quyết định việc đầu tư thì việc lấy ý kiến đại diện cộng đồng dân cư, tổ chức, cá nhân liên quan tiến hành như thế nào và có cần phải công khai thông tin về những nội dung liên quan đến khai thác, sử dụng tài nguyên nước theo quy định?
Average similarity: 0.9208710789680481
-----
Query: Trong trường hợp nào thì khai thác khoáng sản làm vật liệu xây dựng thông thường thông thường không phải xin giấy phép khai thác?
Average similarity: 0.8974435925483704
-----
Query: Hồ sơ đề nghị góp ý đối với hồ sơ thiết kế cơ sở gồm những thà

In [22]:
pip install deepeval

   ---------------------------------------- 0.0/794.9 kB ? eta -:--:--
   ------------- -------------------------- 262.1/794.9 kB ? eta -:--:--
   ---------------------------------------- 794.9/794.9 kB 5.0 MB/s  0:00:00
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 8.6 MB/s  0:00:00
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 10.9 MB/s  0:00:00

   ----------------------------------------  0/10 [tabulate]
   ----------------------------------------  0/10 [tabulate]
   ----------------------------------------  0/10 [tabulate]
   ---- -----------------------------------  1/10 [sentry-sdk]
   ---- -----------------------------------  1/10 [sentry-sdk]
   ---- -----------------------------------  1/10 [sentry-sdk]
   ---- -----------------------------------  1/10 [sentry-sdk]
   ---- -----------------------------------  1/10 [sentry-


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [18]:
from dotenv import load_dotenv
load_dotenv("E:\Downloads\projectcongty\.env")

True

In [25]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models.base_model import DeepEvalBaseLLM
from multi_agent import llm  # Import LLM Gemini của bạn

# 1. Tạo Wrapper cho Custom LLM (Gemini)
class GoogleGeminiWrapper(DeepEvalBaseLLM):
    def __init__(self, model):
        self.model = model

    def load_model(self):
        return self.model

    def generate(self, prompt: str) -> str:
        # DeepEval gửi prompt dưới dạng string, ta gọi invoke của LangChain
        response = self.model.invoke(prompt)
        return response.content

    async def a_generate(self, prompt: str) -> str:
        # Hỗ trợ async nếu cần
        response = await self.model.ainvoke(prompt)
        return response.content

    def get_model_name(self):
        return "gemini-2.5-flash"

# Khởi tạo wrapper
custom_gemini = GoogleGeminiWrapper(llm)

# 2. Định nghĩa Metric GEval cho Query Coverage
# Tiêu chí: Các sub-queries có bao phủ đầy đủ các khía cạnh của câu hỏi gốc không?
query_coverage_metric = GEval(
    name="Query Coverage",
    criteria="Determine whether the generated sub-queries cover all necessary aspects and intents of the original query to ensure a comprehensive search.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=custom_gemini, # Sử dụng LLM của bạn làm giám khảo
    threshold=0.7
)

# 3. Chạy đánh giá trên tập test
print("--- Bắt đầu đánh giá Query Coverage bằng GEval ---")
geval_scores = []

# Chạy thử trên 5 mẫu đầu tiên để tiết kiệm thời gian
for i, (original_query, sub_queries) in enumerate(zip(questions_test[:], sub_queries_list[:])):
    
    # Chuyển list sub-queries thành string để đưa vào actual_output
    # Lưu ý: actual_output thường là câu trả lời, nhưng ở đây ta đang đánh giá output của agent sinh query
    sub_queries_str = "\n".join(sub_queries)
    
    test_case = LLMTestCase(
        input=original_query,
        actual_output=sub_queries_str,
        expected_output="" # Không cần thiết cho GEval nếu tiêu chí không yêu cầu so sánh
    )
    
    # Thực hiện đo lường
    query_coverage_metric.measure(test_case)
    
    print(f"Query {i+1}: {original_query}")
    print(f"Score: {query_coverage_metric.score}")
    print(f"Reason: {query_coverage_metric.reason}")
    print("-" * 30)
    
    geval_scores.append(query_coverage_metric.score)

if geval_scores:
    print(f"Average Query Coverage Score: {sum(geval_scores) / len(geval_scores)}")

--- Bắt đầu đánh giá Query Coverage bằng GEval ---


Query 1: Để xác định trước mã số cần chuẩn bị hồ sơ như thế nào?
Score: 1.0
Reason: The Actual Output comprehensively addresses the Input's primary intent of understanding 'how to prepare the dossier for pre-determining the code'. While the Input specifically asks about dossier preparation, the 'như thế nào?' (how?) implies a need for broader procedural information. The sub-queries effectively break down this 'how' into essential components: conditions, specific documents, overall procedure, responsible authority, and practical details like processing time and cost. This collective set of sub-queries fully represents the Input's search requirements, ensuring no critical components related to the process are missed.
------------------------------


Query 2: Xin cho hỏi thời hạn thanh toán trực tiếp chi phí khám, chữa bệnh bảo hiểm y tế tại cơ quan bảo hiểm xã hội là bao lâu?
Score: 0.9
Reason: The response directly addresses the input's primary intent by including the original query verbatim as the first sub-query. It further enhances the coverage of the 'thời hạn' (time limit) aspect with a specific sub-query about 'thời hạn nộp hồ sơ và thời hạn giải quyết'. While the output includes additional sub-queries about conditions, documents, procedures, and competent authority that were not explicitly requested in the input, these are highly relevant to the overall process of direct payment of health insurance costs and do not miss any critical components of the original search requirement.
------------------------------


Query 3: Thời điểm lập Hồ sơ cấp giấy phép của các công trình thuộc đối tượng lấy ý kiến đại diện cộng đồng dân cư, tổ chức, cá nhân liên quan trong khai thác, sử dụng tài nguyên nước sau khi quyết định việc đầu tư thì việc lấy ý kiến đại diện cộng đồng dân cư, tổ chức, cá nhân liên quan tiến hành như thế nào và có cần phải công khai thông tin về những nội dung liên quan đến khai thác, sử dụng tài nguyên nước theo quy định?
Score: 1.0
Reason: The actual output comprehensively addresses all key aspects and primary intents identified in the input. It breaks down the complex original query regarding the 'how' and 'public disclosure' of community consultation for water resource projects into specific, actionable sub-queries. These sub-queries cover the conditions for consultation, the required documentation, the procedures, the responsible authorities, and the specific timing for both consultation and public disclosure, thereby fully representing the input's search requirements.
----------

Query 4: Trong trường hợp nào thì khai thác khoáng sản làm vật liệu xây dựng thông thường thông thường không phải xin giấy phép khai thác?
Score: 0.9
Reason: The Actual Output successfully identifies and directly addresses the primary intent of the Input query regarding the circumstances under which a mining license is not required for ordinary construction materials. The first two sub-queries directly rephrase this core question. Furthermore, the output provides a comprehensive set of related sub-queries that anticipate follow-up questions, covering definitions, procedures, authority, and penalties, which enhances the overall utility without missing any critical components of the original search requirement.
------------------------------


...

Query 16: Tôi muốn biết sau bao lâu tôi sẽ được gia hạn hiệu lực VBBH?
Score: 1.0
Reason: The Actual Output comprehensively covers the primary intent of 'gia hạn hiệu lực VBBH' from the Input. It directly addresses the 'sau bao lâu' aspect with two sub-queries (the first repeating the input and the last clarifying 'Thời hạn giải quyết hồ sơ'). Beyond that, it provides a thorough set of related sub-queries covering conditions, required documents, procedures, and competent authority, ensuring all critical components for understanding the renewal process are addressed.
------------------------------


Query 17: Điều kiện cấp Giấy chứng nhận đủ điều kiện hoạt động kiểm định kỹ thuật an toàn lao động đối với tổ chức gồm những gì?
Score: 0.7
Reason: The Actual Output successfully addresses the primary intent of the Input by including sub-queries directly asking about the 'Điều kiện cấp Giấy chứng nhận đủ điều kiện hoạt động kiểm định kỹ thuật an toàn lao động đối với tổ chức'. However, it expands beyond the Input's specific search requirement for 'conditions' by adding queries about the application dossier, procedure, competent authority, and validity period. While these are related, they are not explicitly requested by the original query. There is also redundancy in the first two sub-queries.
------------------------------


Query 18: Thành phần hồ sơ và trách nhiệm lập hồ sơ đối với thương binh đang công tác trong Quân đội đề nghị giám định bổ sung vết thương còn sót?
Score: 0.9
Reason: The response effectively breaks down the input query into multiple sub-queries. It explicitly addresses the 'Thành phần hồ sơ' and 'Trách nhiệm lập hồ sơ' aspects from the input through specific sub-queries (3 and 6). Furthermore, it provides a comprehensive set of related sub-queries covering 'Điều kiện', 'Trình tự, thủ tục', and 'Cơ quan có thẩm quyền', which are highly relevant to the primary intent of 'đề nghị giám định bổ sung vết thương còn sót'. The only minor shortcoming is that the first sub-query is a direct restatement of the input, rather than a breakdown, but the subsequent queries demonstrate strong alignment with the evaluation steps.
------------------------------


Query 19: Ngân hàng Nhà nước Chi nhánh tỉnh thành phố có tiếp nhận và xử lý hồ sơ đăng ký khoản cho vay ra nước ngoài của tổ chức kinh tế trên địa bàn không?
Score: 1.0
Reason: The response strongly aligns with the evaluation steps. It accurately identifies the primary intent of the input regarding the provincial branch's authority to process overseas loan applications. Sub-queries 1 and 5 directly address this core question. Furthermore, the output provides a comprehensive set of highly relevant follow-up queries covering conditions, required documents, procedures, and processing times, which fully represent the broader search requirements implied by the initial specific query, ensuring no critical components are missed.
------------------------------


Query 20: Công ty của chúng  tôi mới được  cấp giấy chứng nhận đủ điều kiện  huấn luyện an toàn, vệ sinh lao động hạng B. Nay do Công ty chúng tôi thay đổi tên Công ty nên Công ty đã nộp hồ sơ đề nghị đổi tên Công ty trong giấy chứng nhận đủ điều kiện huấn luyện an toàn, vệ sinh lao động, vậy sau bao nhiêu ngày chúng tôi có thể nhận được giấy chứng nhận đổi tên?
Score: 0.6
Reason: The response correctly identified the primary intent of the input, which was to determine the processing time for the company name change on the certificate. Sub-query 6 directly addresses this intent, and sub-query 1 is a verbatim copy of the original query, also covering the intent. However, the output includes four additional sub-queries (conditions, required documents, procedures, and competent authority) that introduce new search requirements not explicitly present in the input's specific question about processing time. This indicates that the output does not 'fully represent the Input's search requireme

Eval laws answer agent

In [44]:
import pandas as pd

# Login using e.g. `huggingface-cli login` to access this dataset
df = pd.read_parquet("hf://datasets/thangvip/vietnamese-legal-qa/data/train-00000-of-00001.parquet")

In [45]:
df.head(5)

,doc_name,doc_type_name,article_content,generated_qa_pairs,generation_time
0,"Luật Địa chất và Khoáng sản của Quốc hội, số 5...",Luật,Điều 5. Nguyên tắc hội nhập và hợp tác quốc tế...,[{'answer': 'Theo Khoản 1 Điều 5 Luật Địa chất...,11.126226
1,"Luật Địa chất và Khoáng sản của Quốc hội, số 5...",Luật,"Điều 2. Giải thích từ ngữ\n\nTrong Luật này, c...",[{'answer': 'Dựa trên Khoản 1 Điều 2 của Luật ...,11.682184
2,"Luật Địa chất và Khoáng sản của Quốc hội, số 5...",Luật,Điều 1. Phạm vi điều chỉnh\n\n1. Luật này quy ...,[{'answer': 'Theo khoản 1 Điều 1 của Luật Địa ...,11.741774
3,"Luật Địa chất và Khoáng sản của Quốc hội, số 5...",Luật,"Điều 4. Nguyên tắc điều tra cơ bản địa chất, đ...",[{'answer': 'Theo Điều 4 Khoản 1 của Luật Địa ...,16.472521
4,"Luật Địa chất và Khoáng sản của Quốc hội, số 5...",Luật,"Điều 3. Chính sách của Nhà nước về địa chất, k...",[{'answer': 'Theo Khoản 1 Điều 3 của Luật Địa ...,18.148291


In [30]:
from laws_agent import run_laws_agent

In [ ]:
from deepeval.metrics import FaithfulnessMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from langchain_core.messages import HumanMessage
from laws_agent import run_laws_agent
import json
import numpy as np

# 1. Cấu hình Metrics
# Faithfulness: Kiểm tra xem câu trả lời có dựa trên context không
faithfulness_metric = FaithfulnessMetric(
    threshold=0.7,
    model=custom_gemini,
    include_reason=True
)

# Thay thế AnswerCorrectnessMetric bằng GEval với tiêu chí Correctness
correctness_metric = GEval(
    name="Answer Correctness",
    criteria="Determine whether the actual output is factually correct and aligns with the expected output (ground truth).",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    model=custom_gemini,
    threshold=0.7
)

# ...existing code...
print("--- Bắt đầu đánh giá Laws Agent ---")

results = []
question_count = 0
TARGET_SAMPLES = 5

# Duyệt qua dataframe (bỏ head(5) để duyệt đến khi đủ số câu hỏi thì thôi)
for index, row in df.iterrows():
    if question_count >= TARGET_SAMPLES:
        break

    context_content = row['article_content']
    qa_pairs_raw = row['generated_qa_pairs']
    
    # Xử lý format dữ liệu qa_pairs
    qa_pairs = []
    if isinstance(qa_pairs_raw, np.ndarray):
        qa_pairs = qa_pairs_raw.tolist()
    elif isinstance(qa_pairs_raw, list):
        qa_pairs = qa_pairs_raw
    elif isinstance(qa_pairs_raw, str):
        try:
            # Thay thế single quote bằng double quote để parse JSON nếu cần
            qa_pairs = json.loads(qa_pairs_raw)
        except json.JSONDecodeError:
            try:
                qa_pairs = json.loads(qa_pairs_raw.replace("'", '"'))
            except:
                print(f"Không thể parse qa_pairs tại index {index}")
                continue

    # Duyệt qua từng cặp câu hỏi trong list
    for qa in qa_pairs:
        if question_count >= TARGET_SAMPLES:
            break

        question = qa.get('question') 
        ground_truth = qa.get('answer')
        
        if not question or not ground_truth:
            continue

        print(f"\nĐang test câu hỏi {question_count + 1}/{TARGET_SAMPLES}: {question}")

        # 3. Chạy Agent
        state = {
            "messages": [HumanMessage(content=question)],
            "laws_context": context_content
        }
        
        try:
            response_message = run_laws_agent(state)
            actual_output = response_message.content
        except Exception as e:
            print(f"Lỗi khi chạy agent: {e}")
            continue

        # 4. Tạo Test Case
        test_case = LLMTestCase(
            input=question,
            actual_output=actual_output,
            expected_output=ground_truth,
            retrieval_context=[context_content]
        )

        # 5. Đo lường
        try:
            faithfulness_metric.measure(test_case)
            print(f"Faithfulness Score: {faithfulness_metric.score}")
            print(f"Reason: {faithfulness_metric.reason}")
        except Exception as e:
            print(f"Lỗi khi chấm Faithfulness: {e}")
            faithfulness_metric.score = 0

        try:
            correctness_metric.measure(test_case)
            print(f"Correctness Score: {correctness_metric.score}")
            print(f"Reason: {correctness_metric.reason}")
        except Exception as e:
            print(f"Lỗi khi chấm Correctness: {e}")
            correctness_metric.score = 0

        print("-" * 50)

        results.append({
            "question": question,
            "faithfulness": faithfulness_metric.score,
            "correctness": correctness_metric.score,
            "actual_output": actual_output,
            "expected_output": ground_truth
        })
        
        question_count += 1

# Tính điểm trung bình
if results:
    avg_faith = sum(r['faithfulness'] for r in results) / len(results)
    avg_corr = sum(r['correctness'] for r in results) / len(results)
    print(f"\n=== KẾT QUẢ TỔNG HỢP ===")
    print(f"Số lượng test case: {len(results)}")
    print(f"Average Faithfulness: {avg_faith}")
    print(f"Average Correctness: {avg_corr}")

--- Bắt đầu đánh giá Laws Agent ---

Đang test câu hỏi 1/5: Theo Điều 5 Luật Địa chất và Khoáng sản, hội nhập và hợp tác quốc tế về địa chất, khoáng sản được thực hiện trong những hoạt động cụ thể nào?


Faithfulness Score: 1.0
Reason: The score is 1.00 because the actual output is perfectly faithful to the retrieval context, with no contradictions whatsoever. Excellent work!


Correctness Score: 0.9
Reason: The Actual Output is factually accurate and fully aligns with the Expected Output in terms of content, scope, and intent, correctly identifying the four activities from Khoản 1 Điều 5 Luật Địa chất và Khoáng sản. However, it deviates from the Expected Output's concise, single-sentence format by including additional formatting such as a heading, an introductory phrase, bullet points, and brief explanations for each activity, making it less direct than the expected response.
--------------------------------------------------

Đang test câu hỏi 2/5: Theo Điều 5 của Luật, khi thực hiện hội nhập và hợp tác quốc tế về địa chất, khoáng sản, Việt Nam cần tuân thủ những nguyên tắc và khuôn khổ pháp lý nào?


Faithfulness Score: 1.0
Reason: The score is 1.00 because there are no contradictions, indicating the actual output is perfectly faithful to the retrieval context. Excellent work!


Correctness Score: 0.9
Reason: The Actual Output is factually accurate and aligns very well with the Expected Output in terms of content, scope, and intent. It lists all the same general principles and legal frameworks. The structure of the Actual Output with headings and bullet points is clear and comprehensive. The only minor shortcoming is that the Actual Output does not explicitly cite 'Khoản 1 Điều 5' as the source for some principles, unlike the Expected Output, though it does state it's 'Dựa trên các luật được trích dẫn'.
--------------------------------------------------

Đang test câu hỏi 3/5: Giả sử có một tranh chấp quốc tế phát sinh liên quan đến địa chất, khoáng sản mà Việt Nam là một bên, việc giải quyết tranh chấp này sẽ được thực hiện theo nguyên tắc và cơ sở pháp lý nào theo Luật Địa chất và Khoáng sản? Hãy phân tích ý nghĩa của quy định này.


...

Correctness Score: 0.9
Reason: The Actual Output is factually accurate, correctly identifying the principles (peaceful measures) and legal bases (international custom, international law, law of relevant parties) for dispute resolution as outlined in Article 5, Clause 2. It aligns strongly with the Expected Output in content, scope, and intent, providing a clear explanation of the provision and its significance. While the Actual Output does not explicitly reference Clause 1 as the Expected Output does, its discussion of Vietnam's 'chính sách đối ngoại hòa bình' (peaceful foreign policy) captures the essence of that broader principle, making the alignment sufficient.
--------------------------------------------------

Đang test câu hỏi 4/5: Theo Điều 2 của Luật Địa chất và Khoáng sản, "Địa chất" được định nghĩa như thế nào?


Faithfulness Score: 1.0
Reason: The score is 1.00 because the actual output is perfectly faithful to the retrieval context, with no contradictions found! Excellent work!


Correctness Score: 0.7
Reason: The Actual Output correctly provides the definition of 'Địa chất' as found in the Expected Output. However, it deviates by adding extra formatting such as a heading, bullet points, and bolding, which are not present in the Expected Output. Additionally, the Actual Output refers to the source as 'khoản 1 Điều 2 của Luật này', which is less specific than the 'Khoản 1 Điều 2 của Luật Địa chất và Khoáng sản' mentioned in the Expected Output.
--------------------------------------------------

Đang test câu hỏi 5/5: Hãy phân tích sự khác biệt và mối quan hệ giữa "Di chỉ địa chất" và "Di sản địa chất" theo quy định tại Điều 2 của Luật Địa chất và Khoáng sản.


Faithfulness Score: 1.0
Reason: The score is 1.00 because the actual output is perfectly faithful to the retrieval context, with no contradictions whatsoever! Excellent work!


Correctness Score: 1.0
Reason: The actual output is factually accurate, correctly defining 'Di chỉ địa chất' and 'Di sản địa chất' based on Điều 2. It aligns strongly with the expected output in content and intent, providing a more detailed and structured explanation of the differences and relationships, which enhances clarity without introducing any inaccuracies or deviations from the core information.
--------------------------------------------------

=== KẾT QUẢ TỔNG HỢP ===
Số lượng test case: 5
Average Faithfulness: 1.0
Average Correctness: 0.8800000000000001


Docs Answer Agent Eval

In [1]:
from deepeval.metrics import FaithfulnessMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from langchain_core.messages import HumanMessage
from documens_agent import run_document_agent
import json
import numpy as np

In [2]:
from dotenv import load_dotenv
load_dotenv("E:\Downloads\projectcongty\.env")

True

In [3]:
import pandas as pd
import pandas as pd

splits = {'train': 'data/train-00000-of-00001.parquet', 'validation': 'data/validation-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/taidng/UIT-ViQuAD2.0/" + splits["validation"])

In [31]:
df["answers_only"] = df["answers"].apply(lambda x: x["text"][0] if len(x["text"]) > 0 else "")

In [32]:
#loại bỏ các hàng có answers_only = ""
df = df[df["answers_only"] != ""].reset_index(drop=True)

In [33]:
df.head(5)

,id,uit_id,title,context,question,answers,is_impossible,plausible_answers,answers_only
0,0008-0003-0002,uit_001746,Kiến,Kiến được tìm thấy trên tất cả các lục địa trừ...,Loài kiến chiếm 25% tổng sinh khối của hệ động...,"{'text': ['nhiệt đới'], 'answer_start': [639]}",False,None,nhiệt đới
1,0004-0007-0004,uit_001035,Chiến tranh Vùng Vịnh,Với thắng lợi mới đạt được của Iran trong cuộc...,Tổ chức Abu Nidal bị đuổi sang Syria vào thời ...,"{'text': ['tháng 11 năm 1983'], 'answer_start'...",False,None,tháng 11 năm 1983
2,0003-0047-0001,uit_000930,Canada,Canada có hai ngôn ngữ chính thức là tiếng Anh...,Ngôn ngữ nào phổ biến nhất ở Canada?,"{'text': ['tiếng Anh và tiếng Pháp'], 'answer_...",False,None,tiếng Anh và tiếng Pháp
3,0003-0013-0007,uit_000721,Canada,"Theo Hiệp định Paris 1783, Anh Quốc công nhận ...",Khi nào thì tỉnh Québec được chia làm hai vùng?,"{'text': ['1791'], 'answer_start': [339]}",False,None,1791
4,0010-0005-0003,uit_002122,Gia cầm,Ngan nhà và Ngan bướu mũi đã được thuần hóa từ...,Thịt ngan thường được lấy từ loại ngan nào?,"{'text': ['Ngan nhà và Ngan bướu mũi'], 'answe...",False,None,Ngan nhà và Ngan bướu mũi


In [34]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [35]:
first_10_questions = df["question"][:10].to_list()
first_10_context = df["context"][:10].to_list()
first_10_answers = df["answers_only"][:10].to_list()

In [1]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models.base_model import DeepEvalBaseLLM
from multi_agent import llm

# 1. Tạo Wrapper cho Custom LLM (Gemini)
class GoogleGeminiWrapper(DeepEvalBaseLLM):
    def __init__(self, model):
        self.model = model

    def load_model(self):
        return self.model

    def generate(self, prompt: str) -> str:
        # DeepEval gửi prompt dưới dạng string, ta gọi invoke của LangChain
        response = self.model.invoke(prompt)
        return response.content

    async def a_generate(self, prompt: str) -> str:
        # Hỗ trợ async nếu cần
        return self.generate(prompt)

    def get_model_name(self):
        return "gemini-2.5-flash"

# Khởi tạo wrapper
custom_gemini = GoogleGeminiWrapper(llm)

In [37]:
faithfulness_metric = FaithfulnessMetric(
    threshold=0.7,
    model = custom_gemini,
    include_reason=True
)

correctness_metric = GEval(
    name = "Answer Correctness",
    criteria= "Determine whether the actual output is factually correct and aligns with the expected output (ground truth).",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    model = custom_gemini,
    threshold=0.7
)

In [38]:
results = {}
for question,context, expected_answer in zip(first_10_questions, first_10_context, first_10_answers):
    state = {
        "messages": [HumanMessage(content=question)],
        "docs_context": context
    }
    response_message = run_document_agent(state)
    actual_output = response_message.content

    test_case = LLMTestCase(
        input=question,
        actual_output=actual_output,
        expected_output=expected_answer,
        retrieval_context=[context]
    )

    faithfulness_metric.measure(test_case)
    print(f"\nQuestion: {question}")
    print(f"Faithfulness Score: {faithfulness_metric.score}")
    print(f"Reason: {faithfulness_metric.reason}")

    correctness_metric.measure(test_case)
    print(f"Correctness Score: {correctness_metric.score}")
    print(f"Reason: {correctness_metric.reason}")

    print("-" * 50)
    results["faithfulness_scores"] = results.get("faithfulness_scores", []) + [faithfulness_metric.score]
    results["correctness_scores"] = results.get("correctness_scores", []) + [correctness_metric.score]

print(f"\n=== KẾT QUẢ TỔNG HỢP ===")
print(f"Average Faithfulness Score: {sum(results['faithfulness_scores']) / len(results['faithfulness_scores'])}")
print(f"Average Correctness Score: {sum(results['correctness_scores']) / len(results['correctness_scores'])}")


Question: Giải bóng chày Puerto Rico diễn ra vào thời gian nào trong năm?
Faithfulness Score: 1.0
Reason: The score is 1.00 because the actual output is perfectly faithful to the retrieval context, with no contradictions present. Excellent work!


Correctness Score: 0.6
Reason: The Actual Output correctly identifies 'mùa đông' as per the Expected Output, demonstrating factual accuracy. However, it includes significant additional text ('Dựa trên nội dung văn bản nhập vào, giải bóng chày chuyên nghiệp của Puerto Rico được tổ chức vào') which is not present in the concise Expected Output. This constitutes a misalignment in completeness and conciseness, as the Actual Output is much more verbose than expected.
--------------------------------------------------



Question: Vụ việc ly hôn của Eleanor và Roosevelt được giải quyết như thế nào?
Faithfulness Score: 1.0
Reason: The score is 1.00 because the actual output is perfectly faithful to the retrieval context, with no contradictions whatsoever. Excellent work!


Correctness Score: 0.6
Reason: The Actual Output correctly identifies the initial part of the resolution, stating that Eleanor and Roosevelt reconciled 'Qua trung gian của Louis Howe, một cố vấn của Franklin, hai người hòa giải với nhau', which aligns with the Expected Output. However, it fails to include the subsequent crucial detail about Eleanor's living arrangements for the remainder of their marriage, specifically 'nhưng trong thời gian còn lại của cuộc hôn nhân, Eleanor đến sống một mình trong một ngôi nhà ở Hyde Park tại Valkill', indicating a lack of completeness.
--------------------------------------------------



Question: Tại khu vực miền tây Mông Cổ là nơi của người thuộc chủng nào sinh sống vào thời đại đồ đồng đá?
Faithfulness Score: 1.0
Reason: The score is 1.00 because the actual output is perfectly faithful to the retrieval context, with no contradictions whatsoever! Great job!


...

Correctness Score: 0.2
Reason: The actual output includes the expected entity 'Houston' but fails significantly in alignment with the expected output. It provides a verbose sentence with additional, unrequested factual information about population and an introductory phrase, instead of just the concise entity 'Houston'. This demonstrates a major discrepancy in both content and format compared to the expected output.
--------------------------------------------------



Question: Những cách thức nào thường được sử dụng trong các cuộc tấn công bằng bom hoá học?
Faithfulness Score: 1.0
Reason: The score is 1.00 because the actual output is perfectly faithful to the retrieval context, with no contradictions found. Excellent work!


Correctness Score: 0.7
Reason: The Actual Output conveys the same core factual information as the Expected Output. However, it includes an additional introductory phrase, 'Dựa trên nội dung văn bản nhập vào,' which is not present in the Expected Output. This prevents full alignment and exact completeness as per the ground truth, despite the core statement being factually correct.
--------------------------------------------------



Question: Việc Uganda tham chiến tại chiến trường Cộng hòa Congo đã gây ra thách thức nào cho nền kinh tế Uganda?
Faithfulness Score: 1.0
Reason: The score is 1.00 because the actual output is perfectly faithful to the retrieval context, with no contradictions found! Excellent work!


Correctness Score: 0.4
Reason: The Actual Output does not fully align with the Expected Output. It provides a complete sentence rather than the specific short phrase expected, indicating a significant structural misalignment. Additionally, the key phrase in the Actual Output includes the word 'kinh tế' ('economic'), which is an addition not present in the Expected Output's 'duy trì mức tăng trưởng khả quan' ('maintaining favorable growth'), representing a factual discrepancy from the ground truth.
--------------------------------------------------

=== KẾT QUẢ TỔNG HỢP ===
Average Faithfulness Score: 1.0
Average Correctness Score: 0.33999999999999997


Eval Extract Docs Agent

In [2]:
#lấy list tất cả các đường dẫn tuyệt đối file trong thư mục "pdf files"
import os
pdf_folder = r"C:\Users\Admin\Downloads\Project code\pdf files"
pdf_files = [os.path.join(pdf_folder, f) for f in os.listdir(pdf_folder) if f.endswith('.pdf')]
print(pdf_files)

['C:\\Users\\Admin\\Downloads\\Project code\\pdf files\\hdsd mạng xã hội.pdf', 'C:\\Users\\Admin\\Downloads\\Project code\\pdf files\\Kỹ thuật môn bóng đá.pdf', 'C:\\Users\\Admin\\Downloads\\Project code\\pdf files\\nâng cao năng lực nghiên cứu đổi mới sáng tạo của sinh viên.pdf', 'C:\\Users\\Admin\\Downloads\\Project code\\pdf files\\thói quen sd mxh sinh viên.pdf', 'C:\\Users\\Admin\\Downloads\\Project code\\pdf files\\thể chế chính trị cộng hòa.pdf', 'C:\\Users\\Admin\\Downloads\\Project code\\pdf files\\tâm lý tình cảm vị thành niên.pdf', 'C:\\Users\\Admin\\Downloads\\Project code\\pdf files\\áp dụng công nghệ trong thể thao.pdf', 'C:\\Users\\Admin\\Downloads\\Project code\\pdf files\\ô nhiễm môi trường.pdf', 'C:\\Users\\Admin\\Downloads\\Project code\\pdf files\\đổi mới mô hình kinh doanh.pdf', 'C:\\Users\\Admin\\Downloads\\Project code\\pdf files\\ứng dụng AI trong học tập.pdf']


In [5]:
import os
from deepeval import evaluate
from deepeval.metrics import ContextualRecallMetric, ContextualPrecisionMetric
from deepeval.test_case import LLMTestCase
from multi_agent import AgentState
from extract_docs_agent import run_docs_agent 
import pandas as pd
from langchain_core.messages import HumanMessage
import time
test_data = pd.read_csv(r"C:\Users\Admin\Downloads\Project code\dataset\dataset_for_extract_docs.csv")
# Định nghĩa Metrics
context_precision = ContextualPrecisionMetric(threshold=0.7, model=custom_gemini)
context_recall = ContextualRecallMetric(threshold=0.7, model=custom_gemini)

test_cases = []
times = []
print("--- Bắt đầu đánh giá Retrieval ---")
scores = {}
for data in test_data.itertuples():
    query = data.Question
    state = AgentState(messages=[HumanMessage(content=query)], uploaded_files=pdf_files)
    a = time.time()
    retrieved_docs = run_docs_agent(state)
    b = time.time()
    print(f"Thời gian truy xuất tài liệu: {b - a} giây")
    times.append(b - a)
    test_case = LLMTestCase(
        input=data.Question,
        actual_output="Placeholder",
        expected_output=data.Answer, 
        retrieval_context=retrieved_docs 
    )
    test_cases.append(test_case)
    context_recall.measure(test_case)
    context_precision.measure(test_case)
    print(f"Query: {query}")
    print(f"Context Recall Score: {context_recall.score}")
    print(f"Reason: {context_recall.reason}")
    print(f"Context Precision Score: {context_precision.score}")
    print(f"Reason: {context_precision.reason}")
    print("-" * 30)
    scores["recall_score"] = scores.get("recall_score",[]) + [context_recall.score]
    scores["precision_score"] = scores.get("precision_score",[]) + [context_precision.score]

print(f"Average Context Recall Score: {sum(scores["recall_score"])/len(scores["recall_score"])}")
print(f"Average Context Precision Score: {sum(scores["precision_score"])/len(scores["precision_score"])}")
print(f"Average Retrieval Time: {sum(times)/len(times)} giây")

Output()

--- Bắt đầu đánh giá Retrieval ---
Thời gian truy xuất tài liệu: 0.16935420036315918 giây


Output()

Output()

Query: Dựa trên kết quả phân tích hồi quy của nghiên cứu, những nhân tố nào có ảnh hưởng tích cực đến năng lực đổi mới sáng tạo của sinh viên Trường Đại học Hồng Đức, và nhân tố nào được xác định là có tác động lớn nhất?
Context Recall Score: 1.0
Reason: The score is 1.00 because the expected output is perfectly aligned with the information in the nodes in retrieval context, with every sentence (1-6) being fully supported by details from nodes 1, 3, 4, 6, and 10. Excellent work!
Context Precision Score: 0.7166666666666666
Reason: The score is 0.72 because while many relevant nodes in retrieval contexts are ranked highly, the presence of irrelevant nodes at higher positions prevents a perfect score. For instance, the second node in retrieval contexts is irrelevant as it "describes the methodology and preliminary results of reliability and EFA analysis, stating "tác giả sử dụng mô hình hồi quy bội để phân tích ảnh hưởng của c ác nhân tố đến năng lực đổi mới sáng tạo của sinh viên chính q

...

Output()

Output()

Query: Theo nội dung văn bản, những thách thức chính nào đang đặt ra đối với việc phát triển và ứng dụng trí tuệ nhân tạo (AI) trong giáo dục đại học tại Việt Nam liên quan đến đội ngũ giảng viên, người học, hạ tầng kỹ thuật và nguồn nhân lực?
Context Recall Score: 1.0
Reason: The score is 1.00 because the expected output is perfectly supported by the nodes in retrieval context. Excellent work!
Context Precision Score: 1.0
Reason: The score is 1.00 because all relevant nodes are perfectly ranked above all irrelevant nodes, ensuring excellent contextual precision!
------------------------------
Thời gian truy xuất tài liệu: 0.18253874778747559 giây


Output()

Query: Dựa trên nội dung văn bản, hãy trình bày các quy định về số lượng cầu thủ được phép thay thế và quy trình thực hiện việc thay người trong một trận đấu bóng đá.
Context Recall Score: 1.0
Reason: The score is 1.00 because every sentence in the expected output is fully supported by the nodes in retrieval context, with information drawn from nodes 1, 2, and 3. Fantastic work!
Context Precision Score: 1.0
Reason: The score is 1.00 because all relevant nodes are perfectly ranked above all irrelevant nodes, providing an excellent retrieval order!
------------------------------
Average Context Recall Score: 0.8666666666666666
Average Context Precision Score: 0.9433140432098766
Average Retrieval Time: 0.15929808616638183 giây


Laws Answer Agent Eval

In [1]:
import pandas as pd

# Login using e.g. `huggingface-cli login` to access this dataset
df = pd.read_parquet("hf://datasets/thangvip/vietnamese-legal-qa/data/train-00000-of-00001.parquet")

In [9]:
from laws_agent import run_laws_agent
import time
from langchain_core.messages import HumanMessage

In [5]:
df = df.sample(frac=1, random_state=42)

In [11]:
times = []
for i, sample in enumerate(df.itertuples(),1):
    laws_context = (sample.article_content + "\n") *5
    question = sample.generated_qa_pairs[1]['question']
    answer = sample.generated_qa_pairs[1]['answer']
    state = {
        "messages": [HumanMessage(content=question)],
        "laws_context": laws_context}
    a = time.time()
    response = run_laws_agent(state)
    b = time.time()
    print(f"Question {i}: {question}")
    print(f"Time taken: {b - a} seconds")
    times.append(b - a)
    if i==100:
        break
print(f"Average time for each query: {sum(times)/len(times)} seconds")

Question 1: Điều 10 khoản 1 điểm d) quy định về chứng chỉ bồi dưỡng nghiệp vụ hòa giải, đối thoại nhưng cũng nêu rõ các trường hợp được miễn. Hãy giải thích những đối tượng nào được miễn chứng chỉ này và lý do tại sao họ lại được miễn?
Time taken: 3.0841116905212402 seconds
Question 2: Nêu sự khác biệt cơ bản về thủ tục cấp giấy phép sử dụng vũ khí quân dụng giữa đối tượng thuộc phạm vi quản lý của Bộ Quốc phòng và đối tượng không thuộc phạm vi quản lý của Bộ Quốc phòng theo Điều 21?
Time taken: 5.131699085235596 seconds
Question 3: Một tổ chức kinh tế có vốn đầu tư nước ngoài dự định sở hữu 40% vốn điều lệ của một doanh nghiệp trong ngành công nghiệp trọng điểm của Việt Nam. Doanh nghiệp này có vốn điều lệ là 1.800 tỷ đồng. Căn cứ vào Điều 21, việc sở hữu này có cần sự chấp thuận của Thủ tướng Chính phủ hay không? Giải thích.
Time taken: 4.987001895904541 seconds
Question 4: Điều 219 của Luật Tố tụng hành chính quy định như thế nào về trách nhiệm của Tòa án cấp phúc thẩm đối với việc 

Docs Answer Agent Time Eval

In [1]:
import pandas as pd
import pandas as pd

splits = {'train': 'data/train-00000-of-00001.parquet', 'validation': 'data/validation-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/taidng/UIT-ViQuAD2.0/" + splits["validation"])
df["answers_only"] = df["answers"].apply(lambda x: x["text"][0] if len(x["text"]) > 0 else "")
df = df[df["answers_only"] != ""].reset_index(drop=True)

In [3]:
def get_number_of_words(text:str):
    return len(text.split())

df["context_word_count"] = df["context"].apply(get_number_of_words)

In [5]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [18]:
from documens_agent import run_document_agent
import time
from langchain_core.messages import HumanMessage
from multi_agent import AgentState

In [22]:
times = []
for i, sample in enumerate(df.itertuples(),1):
    context = (sample.context + "\n") *17
    question = sample.question
    state = AgentState(messages=[HumanMessage(content=question)], docs_context=context)
    a = time.time()
    response = run_document_agent(state)
    b = time.time()
    print(f"Question {i}: {question}")
    print(f"Time taken: {b - a} seconds")
    print(response.content)
    print("-----")
    times.append(b - a)
    if i==100:
        break
print(f"Average time for each query: {sum(times)/len(times)} seconds")

Question 1: Ngoài Texas thì bang nào tại Hoa Kỳ cùng dẫn đầu trong bảng danh sách các bang có công ty lọt vào Fortune 500?
Time taken: 1.6531167030334473 seconds
Dựa trên nội dung văn bản nhập vào, theo số liệu năm 2010, California đồng hạng với Texas trong việc đứng đầu các tiểu bang có công ty lọt vào Fortune 500.
-----
Question 2: Tình trạng của người điều khiển giao thông đường bộ ở Paris như thế nào?
Time taken: 2.82619047164917 seconds
Dựa trên nội dung văn bản nhập vào, tôi không đủ dữ kiện để đưa ra câu trả lời
-----
Question 3: Mitterrand tham gia chiến trường ở vị trí nào?
Time taken: 1.402839183807373 seconds
Dựa trên nội dung văn bản nhập vào, Mitterrand tham gia chiến trường với vị trí "một trung sĩ bộ binh".
-----
Question 4: Viện Phục hồi Roosevelt Warm Springs được Roosevelt thành lập khi nào?
Time taken: 2.138273239135742 seconds
Dựa trên nội dung văn bản nhập vào, Viện Phục hồi Roosevelt Warm Springs (ban đầu là một trung tâm thủy liệu pháp) được Roosevelt thành lập v

Generate Sub Query Time Eval

In [6]:
import pandas as pd
data = pd.read_csv(r"E:\Downloads\projectcongty\Dataset\sent_truncated_dvc_train.csv", encoding='utf-16')
data = data.sample(frac=1, random_state=42)
from generate_subqueries_agent import run_generate_subqueries_agent
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, SystemMessagePromptTemplate
from langchain_core.messages import HumanMessage
from multi_agent import llm
from pydantic import BaseModel
from multi_agent import AgentState
import time
questions = data["noi_dung_hoi"].tolist()
sub_queries_list = []
questions_test = questions[:50]
times = []
import time 
for query in questions_test:
    state = AgentState(messages=[HumanMessage(content=query)])
    a = time.time()
    sub_queries = run_generate_subqueries_agent(state)
    b = time.time()
    print(f"Time taken for query '{query}': {b - a} seconds")
    print("------------------------------")
    times.append(b - a)

print(f"Average time for each query: {sum(times)/len(times)} seconds")



Time taken for query 'Để xác định trước mã số cần chuẩn bị hồ sơ như thế nào?': 3.5682120323181152 seconds
------------------------------
Time taken for query 'Xin cho hỏi thời hạn thanh toán trực tiếp chi phí khám, chữa bệnh bảo hiểm y tế tại cơ quan bảo hiểm xã hội là bao lâu?': 2.7636053562164307 seconds
------------------------------
Time taken for query 'Thời điểm lập Hồ sơ cấp giấy phép của các công trình thuộc đối tượng lấy ý kiến đại diện cộng đồng dân cư, tổ chức, cá nhân liên quan trong khai thác, sử dụng tài nguyên nước sau khi quyết định việc đầu tư thì việc lấy ý kiến đại diện cộng đồng dân cư, tổ chức, cá nhân liên quan tiến hành như thế nào và có cần phải công khai thông tin về những nội dung liên quan đến khai thác, sử dụng tài nguyên nước theo quy định?': 7.676631689071655 seconds
------------------------------
Time taken for query 'Trong trường hợp nào thì khai thác khoáng sản làm vật liệu xây dựng thông thường thông thường không phải xin giấy phép khai thác?': 4.0966

Extract Law Agent Eval

In [13]:
import json
#read file E:\Downloads\projectcongty\Dataset\laws_100K.json
with open(r"C:\Users\Admin\Downloads\Project code\laws_first_100k.json", "r", encoding='utf-8') as f:
    laws_data = json.load(f)
# Convert to DataFrame
import pandas as pd
laws_df = pd.DataFrame(laws_data)

In [19]:
laws_df.iloc[79152][0]

'Điều 175 34/2016/NĐ-CP quy định chi tiết một số điều và biện pháp thi hành luật ban hành văn bản quy phạm pháp luật Sử dụng chuyên gia 1. Trong quá trình lập đề nghị xây dựng văn bản quy phạm pháp luật, soạn thảo, thẩm định, thẩm tra, Thủ trưởng các cơ quan, tổ chức, đơn vị được sử dụng chuyên gia có năng lực phù hợp với từng công việc. 2. Việc sử dụng chuyên gia phải theo các nguyên tắc sau: a) Được lựa chọn theo tiêu chí cụ thể cho từng công việc; b) Được thuê làm việc theo hợp đồng vụ việc; c) Nếu đã tham gia xây dựng nội dung chính sách, soạn thảo văn bản quy phạm pháp luật thì không tham gia thẩm định, thẩm tra đề nghị xây dựng văn bản quy phạm pháp luật, dự án, dự thảo văn bản quy phạm pháp luật đó. 3. Chuyên gia được hưởng các chế độ sau: a) Được nhận tiền thù lao theo thỏa thuận trong hợp đồng; b) Được cung cấp thông tin có liên quan trong quá trình thực hiện công việc của chuyên gia ghi trong hợp đồng; c) Được hỗ trợ chi phí tham dự hội nghị, hội thảo khoa học trong nước có n

In [2]:
# put the name of the column to rule
column_name = "rule"  # Replace with the actual column name you want to use for rules
def extract_rules_from_laws(df, column_name):
    rules = []
    for index, row in df.iterrows():
        if column_name in row and isinstance(row[column_name], str):
            rules.append(row[column_name])
    return rules
rules = extract_rules_from_laws(laws_df, column_name)

In [3]:
laws_df.rename(columns={0: "rules"}, inplace=True)

In [4]:
#tạo một cột index để đánh số thứ tự các luật
laws_df['index'] = laws_df.index

In [5]:
#chia thành 10 file json, mỗi file json chứa 10000 luật. Đặt tên file là laws_0.json, laws_1.json, ..., laws_9.json. lưu ra thư mục E:\Downloads\projectcongty\Dataset\laws_split
#lưu 2 cột, index (số thứ tự tương ứng của rule trong file gốc laws_100K.json) và rules (nội dung luật)
import os
output_dir = r"C:\Users\Admin\Downloads\Project code\datasets\laws_split"
os.makedirs(output_dir, exist_ok=True)
for i in range(10):
    start_index = i * 10000
    end_index = start_index + 10000
    laws_subset = laws_df.iloc[start_index:end_index]
    output_file = os.path.join(output_dir, f"laws_{i}.json")
    laws_subset.to_json(output_file, orient='records', force_ascii=False, lines=True)
    print(f"Saved {len(laws_subset)} laws to {output_file}")

Saved 10000 laws to C:\Users\Admin\Downloads\Project code\datasets\laws_split\laws_0.json
Saved 10000 laws to C:\Users\Admin\Downloads\Project code\datasets\laws_split\laws_1.json
Saved 10000 laws to C:\Users\Admin\Downloads\Project code\datasets\laws_split\laws_2.json
Saved 10000 laws to C:\Users\Admin\Downloads\Project code\datasets\laws_split\laws_3.json
Saved 10000 laws to C:\Users\Admin\Downloads\Project code\datasets\laws_split\laws_4.json
Saved 10000 laws to C:\Users\Admin\Downloads\Project code\datasets\laws_split\laws_5.json
Saved 10000 laws to C:\Users\Admin\Downloads\Project code\datasets\laws_split\laws_6.json
Saved 10000 laws to C:\Users\Admin\Downloads\Project code\datasets\laws_split\laws_7.json
Saved 10000 laws to C:\Users\Admin\Downloads\Project code\datasets\laws_split\laws_8.json
Saved 10000 laws to C:\Users\Admin\Downloads\Project code\datasets\laws_split\laws_9.json


In [7]:
#đọc lại file laws_0.json để kiểm tra xem đã lưu đúng chưa
import json
with open(r"C:\Users\Admin\Downloads\Project code\dataset\laws_split\laws_9.json", "r", encoding='utf-8') as f:
    laws_2_data = [json.loads(line) for line in f]

import pandas as pd
laws_2_df = pd.DataFrame(laws_2_data)
laws_2_df.head(5)

,rules,index
0,Điều 15 10/2014/NĐ-CP về điều lệ tổ chức và ho...,90000
1,Điều 16 10/2014/NĐ-CP về điều lệ tổ chức và ho...,90001
2,Điều 17 10/2014/NĐ-CP về điều lệ tổ chức và ho...,90002
3,Điều 18 10/2014/NĐ-CP về điều lệ tổ chức và ho...,90003
4,Điều 19 10/2014/NĐ-CP về điều lệ tổ chức và ho...,90004


In [20]:
from retrieve_laws_agent import run_retrieve_laws_agent
from generate_subqueries_agent import run_generate_subqueries_agent
from multi_agent import AgentState
from langchain_core.messages import HumanMessage
import pandas as pd

In [2]:
#test thử run_retrieve_laws_agent
state = AgentState(messages=[HumanMessage(content="Hiến Pháp Việt Nam")])
generated_subqueries = run_generate_subqueries_agent(state)
state["generated_subqueries"] = generated_subqueries
keys = run_retrieve_laws_agent(state)
print(keys)

[699, 1111, 959, 956, 780, 911, 732, 916, 1106, 1049]


In [3]:
data_test = pd.read_csv(r"C:\Users\Admin\Downloads\Project code\dataset\Kết quả Đánh giá model - Dataset trích xuất luật.csv")

In [4]:
data_test.head(5)

,Query,Answer,Lists of laws used
0,Trong công tác quản lý nhà nước và hỗ trợ hoạt...,"Theo quy định hiện hành, Ủy ban nhân dân cấp t...","[2427, 2428]"
1,Hãy so sánh khung hình phạt tù áp dụng cho 'Tộ...,"Theo quy định của Bộ luật Hình sự 2015, mức độ...","[10115, 10116, 10117, 10119]"
2,Một em bé 6 tuổi đang đứng chờ để băng qua một...,"Trong tình huống này, pháp luật quy định cụ th...","[22660, 22661, 29976]"
3,Tôi đang có dự định thành lập một cơ sở trợ gi...,"Để thành lập cơ sở trợ giúp trẻ em, bạn cần đá...","[30000, 30001, 30002]"
4,Ông Nguyễn Văn A là một nhà đầu tư cá nhân. Ôn...,Theo quy định tại khoản 1 Điều 35 Nghị định 15...,"[50084, 50007]"


In [6]:
data_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102 entries, 0 to 101
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Query               102 non-null    object
 1   Answer              102 non-null    object
 2   Lists of laws used  102 non-null    object
dtypes: object(3)
memory usage: 2.5+ KB


In [7]:
queries = data_test["Query"].tolist()

In [8]:
#đổi định dạng cột "List of laws used" từ string thành list
import ast
laws_used = []
for law_list in data_test["Lists of laws used"].tolist():
    laws = ast.literal_eval(law_list)
    laws_used.append(laws)

In [44]:
print(len(laws_used))  #kiểm tra độ dài của laws_used
print(len(queries))  #kiểm tra độ dài của queries

102
102


In [45]:
import time
extracted_laws = []
times = []
for query in queries:
    state = AgentState(messages=[HumanMessage(content=query)])
    state["generated_subqueries"] = run_generate_subqueries_agent(state)
    a = time.time()
    keys = run_retrieve_laws_agent(state)
    b = time.time()
    print(f"Time taken for query: {b - a} seconds")
    times.append(b - a)
    print("------------------------------")
    extracted_laws.append(keys)
print(f"Average time for each query: {sum(times)/len(times)} seconds")

Time taken for query: 0.2276318073272705 seconds
------------------------------
Time taken for query: 0.18805146217346191 seconds
------------------------------
Time taken for query: 0.34168267250061035 seconds
------------------------------
Time taken for query: 0.29288220405578613 seconds
------------------------------
Time taken for query: 0.2828998565673828 seconds
------------------------------
Time taken for query: 0.3659780025482178 seconds
------------------------------
Time taken for query: 0.20931625366210938 seconds
------------------------------
Time taken for query: 0.19351410865783691 seconds
------------------------------
Time taken for query: 0.24759650230407715 seconds
------------------------------
Time taken for query: 0.20422053337097168 seconds
------------------------------
Time taken for query: 0.16798806190490723 seconds
------------------------------
Time taken for query: 0.14406275749206543 seconds
------------------------------
Time taken for query: 0.1438570

In [46]:
print(extracted_laws)
print(laws_used)

[[26113, 5752, 61479, 7610, 24043, 61616, 14040, 14079, 30986, 26120], [9796, 32195, 9768, 33062, 10115, 9809, 33063, 32885, 11398, 7787], [22690, 22649, 20681, 22658, 32145, 22655, 22693, 22657, 22659, 22697], [70593, 9035, 30000, 30004, 70552, 30002, 9023, 70590, 20780, 94635], [79704, 79689, 44871, 79698, 79683, 75318, 77467, 87412, 96120, 77465], [39992, 39967, 39996, 39974, 40011, 39979, 40040, 40067, 40041, 40074], [49755, 57337, 49727, 78334, 57332, 40832, 49728, 70155, 70110, 70136], [80094, 80091, 80065, 80095, 50922, 50924, 80071, 80090, 80066, 80068], [68346, 68348, 68351, 68352, 68353, 19149, 86745, 7598, 13660, 10455], [59554, 59553, 60704, 60734, 60712, 60719, 60692, 60728, 21702, 60706], [26350, 8928, 14076, 14053, 1835, 1834, 1833, 31377, 1798, 1855], [78835, 13335, 13336, 78832, 98410, 13338, 70310, 156, 78844, 70337], [70072, 2934, 2935, 82772, 96977, 75618, 73877, 75754, 8726, 61177], [21531, 21532, 89439, 21533, 21534, 21537, 59678, 86571, 89408, 89410], [84063, 840

In [32]:
from retrieve_laws_agent import retrieve_laws
query = "Hồ sơ khai lệ phí trước bạ đối với tài sản là nhà, đất được nộp tại đâu?"
state = AgentState(messages=[HumanMessage(content=query)])
state["generated_subqueries"] = run_generate_subqueries_agent(state)
a = run_retrieve_laws_agent(state)
print(a)
for i in a:
    print(laws_df.iloc[i][0])

[75529, 42748, 75517, 60850, 52420, 52421, 75532, 88400, 75531, 17304]
Điều 10 140/2016/NĐ-CP về lệ phí trước bạ Khai, thu, nộp lệ phí trước bạ 1. Lệ phí trước bạ được khai theo từng lần phát sinh. Tổ chức, cá nhân có tài sản thuộc đối tượng chịu lệ phí trước bạ có trách nhiệm kê khai và nộp hồ sơ khai lệ phí trước bạ (gồm cả các trường hợp thuộc diện miễn lệ phí trước bạ theo quy định tại Điều 9 Nghị định này) cho Cơ quan Thuế khi đăng ký quyền sở hữu, quyền sử dụng với cơ quan nhà nước có thẩm quyền. 2. Nơi nộp hồ sơ khai lệ phí trước bạ - Đối với tài sản là nhà, đất: Hồ sơ khai lệ phí trước bạ nộp tại cơ quan tiếp nhận hồ sơ về giải quyết thủ tục đăng ký, cấp giấy chứng nhận quyền sử dụng đất, quyền sở hữu nhà và tài sản khác gắn liền với đất theo quy định của pháp luật về đất đai. - Đối với tài sản khác: Hồ sơ khai lệ phí trước bạ nộp tại Chi cục Thuế địa phương nơi đăng ký quyền sở hữu, quyền sử dụng hoặc nộp qua Cổng thông tin điện tử của Tổng cục Thuế đối với hồ sơ khai thuế điệ

In [36]:
from retrieve_laws_agent import retrieve_laws

extracted_laws_2 = []
for query in queries:
    a = retrieve_laws({"query": query})
    extracted_laws_2.append(a)
    print(a)

print(len(extracted_laws_2))


[61479, 14079, 24043, 14007, 15436, 14077, 7610, 31020, 61616, 89468]
[11398, 32195, 9796, 9768, 9818, 32885, 9789, 10333, 33198, 9076]
[22690, 22649, 22655, 22697, 22661, 22658, 22659, 20681, 32148, 22693]
[30002, 20780, 15730, 30000, 20797, 20782, 9023, 20785, 30004, 98964]
[75318, 96120, 50007, 50013, 50034, 50037, 50045, 50047, 50049, 50061]
[39992, 39996, 40011, 40040, 40041, 40042, 40043, 40044, 40045, 40051]
[70155, 70110, 70136, 57266, 57272, 57275, 57276, 57279, 57280, 57281]
[80091, 80094, 80065, 80095, 80105, 80085, 80092, 80063, 80071, 80100]
[68346, 68348, 68351, 68352, 68353, 68354, 68355, 68356, 68357, 68358]
[59554, 60734, 59553, 46326, 60704, 60731, 60719, 94161, 59094, 44372]
[1798, 1855, 1886, 1806, 1811, 1813, 1789, 1807, 1787, 1800]
[98410, 70310, 70337, 98411, 20637, 70314, 4824, 98412, 98418, 70313]
[70072, 96977, 82772, 41345, 22832, 82771, 75618, 70075, 70076, 39963]
[21531, 21532, 21533, 21534, 21537, 21540, 21541, 89412, 89421, 89424]
[84063, 84080, 30907, 30

In [59]:
precisions = []
recalls = []
for output_laws, actual_laws in zip(extracted_laws_2, laws_used):
    output_laws = output_laws[:10]
    tp = set(output_laws) & set(actual_laws)
    fp = set(output_laws) - set(actual_laws)
    fn = set(actual_laws) - set(output_laws)
    precision = len(tp) / (len(tp) + len(fp)) if (len(tp) + len(fp)) > 0 else 0
    recall = len(tp) / (len(tp) + len(fn)) if (len(tp) + len(fn)) > 0 else 0
    print(f"Precision: {precision}, Recall: {recall}")
    precisions.append(precision)
    recalls.append(recall)
    print("-----")
average_precision = sum(precisions) / len(precisions) if precisions else 0
average_recall = sum(recalls) / len(recalls) if recalls else 0
print(f"Average Precision: {average_precision}, Average Recall: {average_recall}")

    

Precision: 0.0, Recall: 0.0
-----
Precision: 0.0, Recall: 0.0
-----
Precision: 0.1, Recall: 0.3333333333333333
-----
Precision: 0.2, Recall: 0.6666666666666666
-----
Precision: 0.1, Recall: 0.5
-----
Precision: 0.0, Recall: 0.0
-----
Precision: 0.1, Recall: 0.5
-----
Precision: 0.1, Recall: 0.5
-----
Precision: 0.0, Recall: 0.0
-----
Precision: 0.1, Recall: 0.5
-----
Precision: 0.0, Recall: 0.0
-----
Precision: 0.0, Recall: 0.0
-----
Precision: 0.0, Recall: 0.0
-----
Precision: 0.0, Recall: 0.0
-----
Precision: 0.1, Recall: 1.0
-----
Precision: 0.0, Recall: 0.0
-----
Precision: 0.0, Recall: 0.0
-----
Precision: 0.0, Recall: 0.0
-----
Precision: 0.0, Recall: 0.0
-----
Precision: 0.0, Recall: 0.0
-----
Precision: 0.0, Recall: 0.0
-----
Precision: 0.0, Recall: 0.0
-----
Precision: 0.1, Recall: 1.0
-----
Precision: 0.0, Recall: 0.0
-----
Precision: 0.0, Recall: 0.0
-----
Precision: 0.1, Recall: 1.0
-----
Precision: 0.0, Recall: 0.0
-----
Precision: 0.1, Recall: 1.0
-----
Precision: 0.0, Re

In [58]:
reciprocal_ranks = []
for output_laws, true_laws in zip(extracted_laws_2, laws_used):
    output_laws = output_laws[:10]
    rank = 0
    for i,law in enumerate(output_laws,1):
        if law in true_laws:
            rank = i
            break
    if rank > 0:
        reciprocal_rank = 1 / rank
        print(f"Reciprocal Rank: {reciprocal_rank}")
        reciprocal_ranks.append(reciprocal_rank)
    else:
        print("Reciprocal Rank: 0")
        reciprocal_ranks.append(0)
    print("-----")
average_reciprocal_rank = sum(reciprocal_ranks) / len(reciprocal_ranks) if reciprocal_ranks else 0
print(f"Average Reciprocal Rank: {average_reciprocal_rank}")


Reciprocal Rank: 0
-----
Reciprocal Rank: 0
-----
Reciprocal Rank: 0.2
-----
Reciprocal Rank: 1.0
-----
Reciprocal Rank: 0.3333333333333333
-----
Reciprocal Rank: 0
-----
Reciprocal Rank: 0.5
-----
Reciprocal Rank: 1.0
-----
Reciprocal Rank: 0
-----
Reciprocal Rank: 0.2
-----
Reciprocal Rank: 0
-----
Reciprocal Rank: 0
-----
Reciprocal Rank: 0
-----
Reciprocal Rank: 0
-----
Reciprocal Rank: 0.16666666666666666
-----
Reciprocal Rank: 0
-----
Reciprocal Rank: 0
-----
Reciprocal Rank: 0
-----
Reciprocal Rank: 0
-----
Reciprocal Rank: 0
-----
Reciprocal Rank: 0
-----
Reciprocal Rank: 0
-----
Reciprocal Rank: 0.1
-----
Reciprocal Rank: 0
-----
Reciprocal Rank: 0
-----
Reciprocal Rank: 0.3333333333333333
-----
Reciprocal Rank: 0
-----
Reciprocal Rank: 1.0
-----
Reciprocal Rank: 0
-----
Reciprocal Rank: 0
-----
Reciprocal Rank: 0
-----
Reciprocal Rank: 0
-----
Reciprocal Rank: 0
-----
Reciprocal Rank: 0
-----
Reciprocal Rank: 0
-----
Reciprocal Rank: 0
-----
Reciprocal Rank: 0
-----
Reciproca

In [57]:
#calculate average precision
average_precisions = []
for output_laws, actual_laws in zip(extracted_laws_2, laws_used):
    output_laws = output_laws[:10]
    total_precision_k = 0
    for i, law in enumerate(output_laws,1):
        first_k_laws = output_laws[:i]
        tp = set(first_k_laws) & set(actual_laws)
        precision_at_k = len(tp) / i if i > 0 else 0
        rel_k = 1 if law in actual_laws else 0
        total_precision_k += precision_at_k * rel_k
    average_precision = total_precision_k/len(actual_laws) if len(actual_laws) > 0 else 0
    print(f"Average Precision: {average_precision}")
    average_precisions.append(average_precision)
    print("-----")

average_of_average_precisions = sum(average_precisions) / len(average_precisions) if average_precisions else 0
print(f"Mean Average Precision (MAP): {average_of_average_precisions}")

Average Precision: 0.0
-----
Average Precision: 0.0
-----
Average Precision: 0.06666666666666667
-----
Average Precision: 0.5
-----
Average Precision: 0.16666666666666666
-----
Average Precision: 0.0
-----
Average Precision: 0.25
-----
Average Precision: 0.5
-----
Average Precision: 0.0
-----
Average Precision: 0.1
-----
Average Precision: 0.0
-----
Average Precision: 0.0
-----
Average Precision: 0.0
-----
Average Precision: 0.0
-----
Average Precision: 0.16666666666666666
-----
Average Precision: 0.0
-----
Average Precision: 0.0
-----
Average Precision: 0.0
-----
Average Precision: 0.0
-----
Average Precision: 0.0
-----
Average Precision: 0.0
-----
Average Precision: 0.0
-----
Average Precision: 0.1
-----
Average Precision: 0.0
-----
Average Precision: 0.0
-----
Average Precision: 0.3333333333333333
-----
Average Precision: 0.0
-----
Average Precision: 1.0
-----
Average Precision: 0.0
-----
Average Precision: 0.0
-----
Average Precision: 0.0
-----
Average Precision: 0.0
-----
Average 

Time Eval Classifer Based Agent

In [19]:
from router_agent import run_router_agent
import pandas as pd
import time as time
from multi_agent import AgentState
from langchain_core.messages import HumanMessage
sentences = pd.read_csv(r"C:\Users\Admin\Downloads\Project code\dataset\results_for_router_matching_based.csv")
sentences = sentences["query"].tolist()
print(len(sentences))
total_time = 0
for i, sentence in enumerate(sentences,1):
    state = {"messages": [HumanMessage(content=sentence)]}
    start_time = time.time()
    route = run_router_agent(state)
    end_time = time.time()
    elapsed_time = end_time - start_time
    total_time += elapsed_time
    print(f"Query {i}: {sentence}")
    print(f"Decided route: {route}")

print(f"Average time per query: {total_time / len(sentences):.4f} seconds")


2000
Query 1: Công ty của chúng  tôi mới được  cấp giấy chứng nhận đủ điều kiện  huấn luyện an toàn, vệ sinh lao động hạng B. Nay do Công ty chúng tôi thay đổi tên Công ty thì chúng tôi có phải làm thủ tục đối tên Công ty trên giấy chứng nhận đã được cấp không? Và thành phần hồ sơ cần phải nộp bao gồm những giấy tờ gì?
Decided route: extract_laws
Query 2: Augustus tự nhận mình là gì?
Decided route: documents
Query 3: Em của tôi đã mất, nay liên quan đến việc phân chia di sản thừa kế của gia đình cần phải có giấy xác nhận việc em tôi còn độc thân. Vậy UBND cấp xã có thẩm quyền cấp giấy này cho em tôi không?
Decided route: extract_laws
Query 4: Nguyên lão có thể bị cách chức nếu vi phạm điều gì?
Decided route: extract_laws
Query 5: Sau khi nộp hồ sơ công nhận hạng cơ sở lưu trú du lịch thì khi nào khách sạn tôi được thẩm định xếp hạng?
Decided route: extract_laws
Query 6: Điều kiện xác nhận nội dung quảng cáo mỹ phẩm là gì?
Decided route: extract_laws
Query 7: Adam Smith dành nhiều thời 